# 📈 TrendSense — Notebook 2: TVI Analysis

**Objective:** Compute Trend Velocity Index (TVI) from Google Trends data, detect viral spikes, and analyze cross-correlation with sales.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import config
from src.data_ingestion import fetch_google_trends, download_walmart_data
from src.tvi import compute_tvi, compute_tvi_acceleration, compute_tvi_features, detect_spikes, get_tvi_summary
from src.utils import plot_tvi_analysis

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
print('✅ Libraries loaded')

## 1. Load Google Trends Data

In [ ]:
trends_df = fetch_google_trends()
print(f'Shape: {trends_df.shape}')
print(f'Date range: {trends_df["date"].min()} to {trends_df["date"].max()}')
keywords = [c for c in trends_df.columns if c != 'date']
print(f'Keywords: {keywords}')
trends_df.head()

## 2. TVI Computation — Example: 'smartphones'

In [ ]:
keyword = keywords[0] if keywords else 'smartphones'
print(f'Analyzing keyword: "{keyword}"')

# Compute full TVI feature set
tvi_feat = compute_tvi_features(trends_df, keyword)
print(f'\nTVI Features shape: {tvi_feat.shape}')
tvi_feat.head(10)

In [ ]:
# TVI Summary
summary = get_tvi_summary(tvi_feat)
print('📊 TVI Summary:')
for k, v in summary.items():
    print(f'   {k}: {v}')

## 3. Three-Panel TVI Visualization

**Panel 1:** Raw Google Trends interest  
**Panel 2:** TVI (rate of change)  
**Panel 3:** Spike detection with severity coloring

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=['Raw Search Interest', 'Trend Velocity Index (TVI)', 'Spike Detection'])

dates = tvi_feat['date']

# Panel 1: Raw trend
fig.add_trace(go.Scatter(x=dates, y=tvi_feat['raw_trend'], mode='lines',
                         name='Search Interest', line=dict(color='#3B82F6', width=2),
                         fill='tozeroy', fillcolor='rgba(59,130,246,0.1)'), row=1, col=1)

# Panel 2: TVI
tvi_vals = tvi_feat['tvi'].fillna(0)
colors = ['#10B981' if v >= 0 else '#EF4444' for v in tvi_vals]
fig.add_trace(go.Bar(x=dates, y=tvi_vals, name='TVI', marker_color=colors, opacity=0.75), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5, row=2, col=1)

# Panel 3: Spikes
for severity, color, size in [('NONE', 'gray', 3), ('MILD', '#F59E0B', 8),
                                ('MODERATE', '#F97316', 12), ('SEVERE', '#EF4444', 18)]:
    mask = tvi_feat['spike_severity'] == severity
    if mask.any():
        fig.add_trace(go.Scatter(x=dates[mask], y=tvi_vals[mask], mode='markers',
                                name=severity, marker=dict(color=color, size=size)), row=3, col=1)

fig.update_layout(template='plotly_dark', height=800, title=f'TVI Analysis — "{keyword}"')
fig.show()

## 4. Multi-Keyword TVI Comparison

In [ ]:
# Compute TVI for all keywords
all_summaries = []
for kw in keywords:
    feat = compute_tvi_features(trends_df, kw)
    s = get_tvi_summary(feat)
    s['keyword'] = kw
    s['category'] = config.TREND_KEYWORDS.get(kw, 'Unknown')
    all_summaries.append(s)

summary_df = pd.DataFrame(all_summaries)
print('📊 All Keywords TVI Summary:')
summary_df[['keyword', 'category', 'mean_tvi', 'std_tvi', 'total_spikes', 'severe_spikes', 'spike_rate_pct']]

## 5. Festive Season Spike Analysis

In [ ]:
# Analyze spikes during festive season (Oct-Nov)
for kw in keywords[:4]:
    feat = compute_tvi_features(trends_df, kw)
    festive = feat[(feat['date'].dt.month >= 10) & (feat['date'].dt.month <= 11)]
    festive_spikes = festive[festive['is_spike']]
    
    print(f'\n🎆 "{kw}" — Festive Season Analysis:')
    print(f'   Total festive weeks: {len(festive)}')
    print(f'   Spikes in festive season: {len(festive_spikes)}')
    if len(festive) > 0:
        print(f'   Festive spike rate: {len(festive_spikes)/len(festive)*100:.1f}%')
        print(f'   Avg festive TVI: {festive["tvi"].mean():.2f}% vs overall {feat["tvi"].mean():.2f}%')

## 6. Cross-Correlation: TVI vs Sales

In [ ]:
from src.lag_optimizer import compute_cross_correlation, find_optimal_lag

# Load sales data
try:
    walmart_df = download_walmart_data()
    weekly_sales = walmart_df.groupby('Date')['Weekly_Sales'].mean().sort_index()
except:
    # Synthetic fallback
    np.random.seed(42)
    weekly_sales = pd.Series(np.random.uniform(10000, 50000, len(trends_df)), name='Weekly_Sales')

# Cross-correlation for first keyword
kw = keywords[0]
tvi = compute_tvi(trends_df[kw])
min_len = min(len(tvi), len(weekly_sales))

corr_df = compute_cross_correlation(tvi.iloc[:min_len].reset_index(drop=True),
                                     weekly_sales.iloc[:min_len].reset_index(drop=True),
                                     max_lag=12)

print(f'Cross-correlation results for "{kw}":')
print(corr_df.to_string())

# Plot
fig = go.Figure()
colors = ['#10B981' if s else '#6366F1' for s in corr_df['is_significant']]
fig.add_trace(go.Bar(x=corr_df['lag_weeks'], y=corr_df['correlation'],
                     marker_color=colors, opacity=0.85))
best_lag = corr_df.loc[corr_df['correlation'].abs().idxmax(), 'lag_weeks']
fig.add_vline(x=best_lag, line_dash='dash', line_color='red',
              annotation_text=f'Optimal: {best_lag}w')
fig.update_layout(template='plotly_dark', title=f'TVI-Sales Cross-Correlation — "{kw}"',
                  xaxis_title='Lag (weeks)', yaxis_title='Correlation', height=400)
fig.show()

## Key Findings

1. **TVI captures acceleration** — not just raw volume
2. **Festive season** shows elevated spike rates
3. **Cross-correlation** reveals optimal lag between trend signals and sales

### Next: Feature Engineering → **Notebook 03**